In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [18]:
# Claude code to learn embeddings
#
# 
#  toy vocab: 5 characters
char_to_idx = {'a': 0, 'b': 1, 'c': 2, '<pad>': 3, '<unk>': 4}
vocab_size = len(char_to_idx)
d_model = 4  # tiny for illustration

embed = nn.Embedding(vocab_size, d_model, padding_idx=char_to_idx['<pad>'])

print("weight matrix (one row per token):")
print(embed.weight)
# shape: (vocab_size, d_model) -> (5, 4)

# encode the word "cab" as indices
word = "abc"
ids = torch.tensor([char_to_idx[c] for c in word])  # shape (3,)
print("\nids:", ids)

vecs = embed(ids)  # shape (3, 4) — one embedding row per character
print("\nembeddings for 'abc':")
print(vecs)

# confirm it's literally just indexing the weight matrix
print("\nsame as manual lookup:", torch.equal(vecs, embed.weight[ids]))

# batch of two words, padded to same length
batch_ids = torch.tensor([
    [char_to_idx['a'], char_to_idx['b'], char_to_idx['c']],
    [char_to_idx['a'], char_to_idx['b'], char_to_idx['<pad>']],
])
batch_vecs = embed(batch_ids)
print("\nbatch shape:", batch_vecs.shape)  # (2, 3, 4) = (batch, seq_len, d_model)
print(batch_vecs)
print("pad row is zero:", torch.all(batch_vecs[1, 2] == 0).item())

weight matrix (one row per token):
Parameter containing:
tensor([[ 0.9522,  0.1625, -1.0632, -1.3267],
        [-0.2042,  0.8504,  1.8104, -0.3015],
        [-1.2295, -1.9304, -0.8417,  0.9061],
        [ 0.0000,  0.0000,  0.0000,  0.0000],
        [-1.2534, -0.6184, -0.9187, -0.1607]], requires_grad=True)

ids: tensor([0, 1, 2])

embeddings for 'abc':
tensor([[ 0.9522,  0.1625, -1.0632, -1.3267],
        [-0.2042,  0.8504,  1.8104, -0.3015],
        [-1.2295, -1.9304, -0.8417,  0.9061]], grad_fn=<EmbeddingBackward0>)

same as manual lookup: True

batch shape: torch.Size([2, 3, 4])
tensor([[[ 0.9522,  0.1625, -1.0632, -1.3267],
         [-0.2042,  0.8504,  1.8104, -0.3015],
         [-1.2295, -1.9304, -0.8417,  0.9061]],

        [[ 0.9522,  0.1625, -1.0632, -1.3267],
         [-0.2042,  0.8504,  1.8104, -0.3015],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>)
pad row is zero: True


In [25]:
italian_words = []
spanish_words = []

with open("it_es_cognates.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        it, es = line.split(";")
        italian_words.append(it.lower())
        spanish_words.append(es.lower())

print(len(italian_words), "pairs")
print(italian_words[:5], spanish_words[:5])

# --- vocab building (from before) ---
all_chars = set()
for word in spanish_words + italian_words:
    all_chars.update(word)

all_chars = sorted(all_chars)

# special caracters: pad, start of string, end of string, unknown
specials = ['<pad>', '<sos>', '<eos>', '<unk>']
vocab = specials + all_chars

char_to_idx = {ch: i for i, ch in enumerate(vocab)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print(vocab)
print(len(vocab))

# test word size in the vocabulary (need to know the dimension of the words I am going to have in the model - by padding)
# I will add 2 or 3 just to be on the safe side for future additions to the dictionary
print("Max length of spanish words: ", max(len(w) for w in spanish_words))
print("Max length of italian words: ", max(len(w) for w in italian_words))

def encode_source(word, max_len = 20):
    # input: no sos/eos needed
    ids = [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len]
    ids += [char_to_idx['<pad>']] * (max_len - len(ids))
    return ids

def encode_target(word, max_len = 20):
    # output: needs sos/eos since decoder generates it step by step
    ids = [char_to_idx['<sos>']] + [char_to_idx.get(c, char_to_idx['<unk>']) for c in word][:max_len-2] + [char_to_idx['<eos>']]
    ids += [char_to_idx['<pad>']] * (max_len - len(ids))
    return ids

print(encode_target('casa'))
print(encode_source('casa'))

884 pairs
['acqua', 'aglio', 'aiutare', 'aiutata', 'aiutate'] ['agua', 'ajo', 'ayudar', 'ayudada', 'ayudadas']
['<pad>', '<sos>', '<eos>', '<unk>', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'x', 'y', 'z', 'à', 'á', 'é', 'í', 'ñ', 'ó', 'ù', 'ú']
36
Max length of spanish words:  15
Max length of italian words:  13
[1, 6, 4, 21, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[6, 4, 21, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
